
<div class="alert alert-block alert-info">  
    
## Import & Connections
</div>

In [1]:
import numpy as np
import pandas as pd
import hashlib
import torch
import glob
import time
import os
import re
import tensorflow as tf
import matplotlib.pyplot as plt

# from nltk.corpus import stopwords
from transformers import pipeline
from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

2025-01-24 10:26:10.939777: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 50)

In [3]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
start_date = '20250120'
end_date = '20250122'
city = 'Bangalore'

In [5]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/ads-monetisation/metadata'
print(local_dataset)

/Users/E2074/analytics/customer/Ads-Monetisation/metadata-analysis
/Users/E2074/local-datasets/customer/ads-monetisation/metadata


<div class="alert alert-block alert-warning">  
    
## Datasets & Functions
</div>

In [6]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

# Generate the list of dates
date_range = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]
print(date_range)

['20250120', '20250121', '20250122']


In [7]:
def get_customer_metadata(date_range, city):

    all_df = []

    title_pattern = 'title[=:]"?([^",}]*)'
    body_pattern = 'body[=:]"?([^",}]*)'
    advertiserName_pattern = 'advertiserName[=:]"?([^",}]*)'
    
    for date in date_range:
    
        customer_metadata = f"""
        
            WITH metadata as (
            SELECT
                userid,
                aduuid,
                title,
                body,
                advertiserName
            FROM 
                (
                SELECT
                    yyyymmdd,
                    userid,
                    COALESCE(aduuid, adid) aduuid,
                    LENGTH(metadata1) metabase_length,
                    
                    CASE 
                    WHEN os = 'android' THEN regexp_extract(metadata1, '{title_pattern}' , 1) 
                    WHEN os = 'iOS' THEN json_extract_scalar(metadata1, '$.title') 
                    END AS title,
                    
                    CASE 
                    WHEN os = 'android' THEN regexp_extract(metadata1, '{body_pattern}', 1)
                    WHEN os = 'iOS' THEN json_extract_scalar(metadata1, '$.body')
                    END AS body,
                    
                    CASE 
                    WHEN os = 'android' THEN regexp_extract(metadata1, '{advertiserName_pattern}', 1)
                    WHEN os = 'iOS' THEN json_extract_scalar(metadata1, '$.advertiserName')
                    END AS advertiserName
                    
                    
                FROM 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                WHERE 
                    yyyymmdd = '{date}'
                    AND city = '{city}'
                    AND responseType = 'GAMBanner'
                    AND os = 'android'
                    AND eventName = 'Ad_Response'
                    AND length(metadata1) > 5
                )
            WHERE
                (title IS NOT NULL OR body IS NOT NULL OR advertiserName IS NOT NULL)
            )
            
            SELECT
                *
            FROM 
                metadata
            
        """
        
        df = pd.read_sql(customer_metadata, connection)
        all_df.append(df)
        df.to_parquet(local_dataset + '/data_dump/response_metadata_{}.parquet'.format(date), index=False)
    
    # Concatenate all DataFrames
    final_df = pd.concat(all_df, ignore_index=True)

    return final_df


# df_data = get_customer_metadata(date_range, city)

In [8]:
def get_customer_base(start_date, end_date, city):
    
    customer_metadata = f"""
    
        with customers as (

            select
                distinct 
                customer_id,
                case 
                when customer_obj_gender = '0' then 'Male' 
                when customer_obj_gender = '1' then 'Female' 
                else 'Other/Unknow' end gender
            from 
                orders.order_logs_fact
            where 
                yyyymmdd >= '{start_date}'
                and yyyymmdd <= '{end_date}'
                and city_name = 'Bangalore'
            ),
            
            segment as (
        
            select
                customer_id,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_income_segment,
                ps_tag_link,
                ps_tag_auto,
                taxi_regularity_segment
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-01-20'
            ),
            
            
            app_cat as (
            
            select 
                *
            from 
                reports.sql_ingestion_customer_appography_category_view
            where 
                yyyymmdd = '20250112'
            ),
            
            age as (
    
            select 
                customer_id,
                age_group,
                confidence_tag
            from 
                hive.experiments_internal.customer_predicated_age_snapshot
            )    
            
            
            select 
                customers.*,
                COALESCE(taxi_ltr_segment, 'UNKNOWN') taxi_ltr_segment,
                COALESCE(taxi_retention_segments, 'UNKNOWN') taxi_retention_segments,
                COALESCE(taxi_regularity_segment, 'UNKNOWN') taxi_regularity_segment,
                COALESCE(taxi_income_segment, 'UNKNOWN') taxi_income_segment,
                ps_tag_link,
                ps_tag_auto,
                ai,
                camerafilter,
                courier,
                dating,
                deliveryfood,
                deliverygrocery,
                devotional,
                driverapp,
                ecommerce,
                educational,
                entertainment,
                fantasysports,
                financeinvestment,
                financenews,
                financetransactions,
                gaming,
                government,
                health,
                homesecurity,
                insurance,
                jobsearch,
                news,
                office,
                ott,
                social,
                streamingmusic,
                telecom,
                tools,
                travelbookings,
                travelmetro,
                travelridehailing,
                vehicle,
                wearable,
                age_group,
                confidence_tag
            from 
                customers
            join 
                app_cat
                on customers.customer_id = app_cat.userid
            left join 
                segment
                on customers.customer_id = segment.customer_id
            left join 
                age
                on customers.customer_id = age.customer_id
        
    """
        
    df = pd.read_sql(customer_metadata, connection)
    df.to_parquet(local_dataset + '/gross_customers_v1.parquet', index=False)
    
    return df


# df_customer_base = get_customer_base(start_date, end_date, city)

In [9]:
def get_customer_metadata():
    
    # Use glob to find all Parquet files in the directory
    parquet_files = glob.glob(os.path.join(local_dataset + '/data_dump/', "*.parquet"))
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    
    return df

df_data = get_customer_metadata()

In [10]:
df_gross_customer = pd.read_parquet(local_dataset + '/gross_customers_v1.parquet')

In [11]:
# Generate a unique hash identifier for each combination
df_data["hash_id"] = df_data.apply(lambda row: hashlib.md5(f"{row['title']}_{row['body']}_{row['advertiserName']}".encode()).hexdigest(), axis=1)
df_data["metadata"] = df_data.apply(lambda row: f"advertiserName: {row['advertiserName']}, title: {row['title']}, body: {row['body']}", axis=1)

In [12]:
df_data.head(1)

,userid,aduuid,title,body,advertiserName,hash_id,metadata
0,66490572ff34137abeffd971,85eaa9ee-29c3-4702-b5ae-aa7f8b666f6866490572ff...,Swiggy: Food Instamart Dineout,,,b1f93384567af2e081aeb23f3ba6a522,"advertiserName: , title: Swiggy: Food Instamar..."


In [13]:
df_gross_customer.head(1)

,customer_id,gender,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,ps_tag_link,ps_tag_auto,ai,camerafilter,courier,dating,deliveryfood,deliverygrocery,devotional,driverapp,ecommerce,educational,entertainment,fantasysports,financeinvestment,financenews,financetransactions,gaming,government,health,homesecurity,insurance,jobsearch,news,office,ott,social,streamingmusic,telecom,tools,travelbookings,travelmetro,travelridehailing,vehicle,wearable,age_group,confidence_tag
0,5906fcc1f2e7f4dd2886c934,Male,PHH,ELITE,DAILY,HIGH_INCOME,UNKNOWN,NPS,0,0,1,0,1,1,0,1,1,1,1,1,1,0,1,0,0,0,1,0,1,0,1,1,1,1,0,1,1,0,1,1,1,26-30,VERY_HIGH


In [14]:
df_metadata = df_data[["hash_id", "advertiserName", "title", "body", "metadata"]].drop_duplicates()
df_metadata.head(1)

,hash_id,advertiserName,title,body,metadata
0,b1f93384567af2e081aeb23f3ba6a522,,Swiggy: Food Instamart Dineout,,"advertiserName: , title: Swiggy: Food Instamar..."


In [15]:
print(df_data.shape)
print(df_gross_customer.shape)
print(df_metadata.shape)

(1559948, 7)
(639955, 43)
(7162, 5)


In [16]:
print(df_data.userid.nunique())
print(df_gross_customer.customer_id.nunique())

363932
639907


<div class="alert alert-block alert-success">  
    
## Metadata Classification 
</div>

In [17]:
# Load the pre-trained model and tokenizer
classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-1")

# Define categories
categories = [
    'Finance','Technology','Entertainment','Automotive','Healthcare','Retail/E-commerce','Travel & Tourism',
    'Food & Beverage', 'Real Estate','Education','Beauty & Personal Care','Consumer Goods','Telecommunications',
    'Sports & Recreation', 'Social Network & Media','Fashion & Apparel','Home & Garden','Legal Services',
    'Logistics & Delivery Services', 'Gaming & Esports','Media & Publishing','Business Services','Energy & Utilities',
    'Agriculture','Luxury Goods & Services', 'Events & Conferences','Parenting & Childcare','Pets & Animals',
    'DIY & Crafts','Spirituality & Religion', 'Automobile', 'Banking', 'Investments', 'Trading', 'Unknown'
]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [18]:
df_metadata.head(1)

,hash_id,advertiserName,title,body,metadata
0,b1f93384567af2e081aeb23f3ba6a522,,Swiggy: Food Instamart Dineout,,"advertiserName: , title: Swiggy: Food Instamar..."


In [19]:
# Function to classify text and add category columns
def classify_text(input_data, classifier, categories):
    if isinstance(input_data, str):  # Handle single-column input
        text = input_data or ''
        return classifier(text, candidate_labels=categories)['labels'][0] if text.strip() else 'Unknown'
    else:  # Handle row input
        categories_dict = {}
        for field in ['advertiserName', 'title', 'body']:
            text = input_data[field] or ''
            categories_dict[f'category_{field}'] = (
                classifier(text, candidate_labels=categories)['labels'][0]
                if text.strip() else 'Unknown'
            )
        return pd.Series(categories_dict)

def get_classification_tag(df):
    # Fill missing values in the necessary columns
    df[['advertiserName', 'title', 'body']] = df[['advertiserName', 'title', 'body']].fillna('')

    # Initialize category columns with default values
    df['category_advertiserName'] = df['advertiserName'].apply(lambda x: classify_text(x, classifier, categories) if x else 'Unknown')
    df['category_title'] = 'Unknown'  # Set default values
    df['category_body'] = 'Unknown'

    # Apply classification for title and body only if advertiserName category is 'Unknown'
    mask_title = df['category_advertiserName'] == 'Unknown'
    df.loc[mask_title, 'category_title'] = df.loc[mask_title, 'title'].apply(
        lambda x: classify_text(x, classifier, categories) if x else 'Unknown'
    )

    mask_body = (df['category_advertiserName'] == 'Unknown') & (df['category_title'] == 'Unknown')
    df.loc[mask_body, 'category_body'] = df.loc[mask_body, 'body'].apply(
        lambda x: classify_text(x, classifier, categories) if x else 'Unknown'
    )

    # Combine categories based on conditions
    df['category'] = df.apply(
        lambda row: row['category_advertiserName'] if row['category_advertiserName'] != 'Unknown'
        else (row['category_title'] if row['category_title'] != 'Unknown' else row['category_body']),
        axis=1
    )

    return df[['hash_id', 'advertiserName', 'title', 'body', 'category']]

In [21]:
# start_time = time.time()

# df_category = get_classification_tag(df_metadata)

# df_category.to_parquet(local_dataset + '/hash_id_category_v1.parquet', index=False)

# end_time = time.time()
# execution_time = (end_time - start_time)/60
# print(f"Execution time: {execution_time} mins")


def process_and_save_chunks(df_metadata, chunk_size=50, output_dir= local_dataset +'/category_dump/'):

  if not os.path.exists(output_dir):
    os.makedirs(output_dir)

  for i in range(0, len(df_metadata), chunk_size):
    start_time = time.time()
    chunk = df_metadata[i:i+chunk_size]
    df_category = get_classification_tag(chunk) 
    file_name = f'hash_id_category_v{i//chunk_size+1}.parquet' 
    file_path = os.path.join(output_dir, file_name)
    df_category.to_parquet(file_path, index=False)
    end_time = time.time()
    execution_time = (end_time - start_time)/60
    print(f"Execution time for chunk {i//chunk_size+1}: {execution_time} mins")

In [22]:
process_and_save_chunks(df_metadata)

Execution time for chunk 1: 1.5055625319480896 mins
Execution time for chunk 2: 1.5999649484952292 mins
Execution time for chunk 3: 1.651532737414042 mins
Execution time for chunk 4: 1.681137748559316 mins
Execution time for chunk 5: 2.756667498747508 mins
Execution time for chunk 6: 4.878777301311493 mins
Execution time for chunk 7: 5.02293910185496 mins
Execution time for chunk 8: 5.6887709657351175 mins
Execution time for chunk 9: 5.46604543129603 mins
Execution time for chunk 10: 8.247049915790559 mins
Execution time for chunk 11: 1.6800481836001078 mins
Execution time for chunk 12: 1.8451650023460389 mins
Execution time for chunk 13: 1.5763756513595581 mins
Execution time for chunk 14: 1.583162748813629 mins
Execution time for chunk 15: 2.0554221312205 mins
Execution time for chunk 16: 2.004720151424408 mins
Execution time for chunk 17: 1.969323166211446 mins
Execution time for chunk 18: 1.9664961496988933 mins
Execution time for chunk 19: 1.997252055009206 mins
Execution time for

In [23]:
def get_metadata_category():
    
    # Use glob to find all Parquet files in the directory
    parquet_files = glob.glob(os.path.join(local_dataset + '/category_dump/', "*.parquet"))
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    
    return df

df_category = get_metadata_category()

In [24]:
df_category.head(1)

,hash_id,advertiserName,title,body,category
0,8bf4cf10f139c08f50c0042e965d7f63,,Prime Video,New On Prime - Bloody Beggar A beggar's life t...,Entertainment


In [25]:
df_gross_customer.head(1)

,customer_id,gender,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,ps_tag_link,ps_tag_auto,ai,camerafilter,courier,dating,deliveryfood,deliverygrocery,devotional,driverapp,ecommerce,educational,entertainment,fantasysports,financeinvestment,financenews,financetransactions,gaming,government,health,homesecurity,insurance,jobsearch,news,office,ott,social,streamingmusic,telecom,tools,travelbookings,travelmetro,travelridehailing,vehicle,wearable,age_group,confidence_tag
0,5906fcc1f2e7f4dd2886c934,Male,PHH,ELITE,DAILY,HIGH_INCOME,UNKNOWN,NPS,0,0,1,0,1,1,0,1,1,1,1,1,1,0,1,0,0,0,1,0,1,0,1,1,1,1,0,1,1,0,1,1,1,26-30,VERY_HIGH


In [26]:
df_data.head(1)

,userid,aduuid,title,body,advertiserName,hash_id,metadata
0,66490572ff34137abeffd971,85eaa9ee-29c3-4702-b5ae-aa7f8b666f6866490572ff...,Swiggy: Food Instamart Dineout,,,b1f93384567af2e081aeb23f3ba6a522,"advertiserName: , title: Swiggy: Food Instamar..."


In [27]:
df_mergev1 = pd.merge(df_data, df_gross_customer, how = 'left', left_on = ['userid'], right_on = ['customer_id'])
df_merge = pd.merge(df_mergev1, df_category[['hash_id', 'category']], how='inner', on ='hash_id')

In [28]:
df_merge.head(1)

,userid,aduuid,title,body,advertiserName,hash_id,metadata,customer_id,gender,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,ps_tag_link,ps_tag_auto,ai,camerafilter,courier,dating,deliveryfood,deliverygrocery,devotional,driverapp,ecommerce,educational,...,fantasysports,financeinvestment,financenews,financetransactions,gaming,government,health,homesecurity,insurance,jobsearch,news,office,ott,social,streamingmusic,telecom,tools,travelbookings,travelmetro,travelridehailing,vehicle,wearable,age_group,confidence_tag,category
0,66490572ff34137abeffd971,85eaa9ee-29c3-4702-b5ae-aa7f8b666f6866490572ff...,Swiggy: Food Instamart Dineout,,,b1f93384567af2e081aeb23f3ba6a522,"advertiserName: , title: Swiggy: Food Instamar...",66490572ff34137abeffd971,Male,PHH,GOLD,BI_WEEKLY,MEDIUM_INCOME,UNKNOWN,NPS,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,None,None,Food & Beverage


In [29]:
df_merge.shape

(1560185, 51)

In [30]:
df = df_merge.copy()
df.shape

(1560185, 51)

<div class="alert alert-block alert-success">  
    
## Analysis 
</div>

In [31]:
df.groupby(['gender','category']).agg(customers = ('userid', 'nunique')).reset_index()\
.pivot(index=['gender'], columns=['category'], values=['customers']).reset_index()

gender   customers                                 \
category               Agriculture Automobile Automotive  Banking   
0               Female       313.0    70830.0      220.0  32307.0   
1                 Male       638.0   132540.0      196.0  57671.0   
2         Other/Unknow        18.0     6075.0       15.0   2612.0   

                                                                               \
category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                         485.0               7.0          318.0        836.0   
1                          84.0               5.0          334.0       1553.0   
2                          29.0               NaN           19.0         72.0   

                                                                               \
category Education Energy & Utilities Entertainment Fashion & Apparel Finance   
0              7.0                3.0       34614.0             214.0   148.0   
1             18.0                3.0       51904.0              63.0   405.0   
2              NaN                NaN        2554.0              11.0     8.0   

                                                                    \
category Food & Beverage Gaming & Esports Healthcare Home & Garden   
0                 5575.0           2260.0       13.0          12.0   
1                 8955.0           4117.0        4.0           7.0   
2                  445.0            172.0        NaN           2.0   

                                                                   \
category Investments Legal Services Logistics & Delivery Services   
0              455.0        20950.0                           5.0   
1             1594.0        47917.0                          11.0   
2               50.0         1957.0                           NaN   

                                                                           \
category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                          409.0                NaN                 576.0   
1                          605.0                7.0                 247.0   
2                           21.0                NaN                  21.0   

                                                                              \
category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0                   4.0       158.0            3443.0                18105.0   
1                   5.0       295.0            3004.0                29051.0   
2                   1.0        23.0             222.0                 1424.0   

                                                                 \
category Spirituality & Religion Sports & Recreation Technology   
0                           15.0                 NaN      597.0   
1                           22.0                 2.0     1669.0   
2                            1.0                 NaN       68.0   

                                                               
category Telecommunications  Trading Travel & Tourism Unknown  
0                      74.0   9635.0            887.0  1887.0  
1                     132.0  30827.0           1587.0  5934.0  
2                       4.0   1042.0             80.0   198.0

In [32]:
df.groupby(['gender']).agg(customers = ('userid', 'nunique')).reset_index()
# .pivot(index=['gender'], columns=['category'], values=['customers'])\
# .to_clipboard(index=False)

,gender,customers
0,Female,119399
1,Male,219297
2,Other/Unknow,10009


In [33]:
df.groupby(['taxi_income_segment']).agg(customers = ('userid', 'nunique')).reset_index()

,taxi_income_segment,customers
0,HIGH_INCOME,166277
1,LOW_INCOME,18584
2,MEDIUM_INCOME,104842
3,UNKNOWN,58968


In [34]:
df.groupby(['taxi_income_segment','category']).agg(customers = ('userid', 'nunique')).reset_index()\
.pivot(index=['taxi_income_segment'], columns=['category'], values=['customers'])\
.reset_index()


taxi_income_segment   customers                                 \
category                     Agriculture Automobile Automotive  Banking   
0                HIGH_INCOME       557.0   101382.0      235.0  46603.0   
1                 LOW_INCOME        12.0    10830.0       14.0   4293.0   
2              MEDIUM_INCOME        94.0    65252.0      116.0  27841.0   
3                    UNKNOWN       306.0    31958.0       66.0  13847.0   

                                                                               \
category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                         369.0               4.0          411.0        442.0   
1                          14.0               NaN           12.0        282.0   
2                         135.0               7.0          126.0        543.0   
3                          80.0               1.0          122.0       1193.0   

                                                                               \
category Education Energy & Utilities Entertainment Fashion & Apparel Finance   
0             13.0                6.0       34749.0             214.0   307.0   
1              NaN                NaN        6494.0               3.0    18.0   
2              8.0                NaN       24911.0              38.0   147.0   
3              4.0                NaN       22910.0              33.0    89.0   

                                                                    \
category Food & Beverage Gaming & Esports Healthcare Home & Garden   
0                 7364.0           2896.0        6.0          12.0   
1                  658.0            344.0        1.0           2.0   
2                 4425.0           1344.0        7.0           4.0   
3                 2527.0           1964.0        3.0           3.0   

                                                                   \
category Investments Legal Services Logistics & Delivery Services   
0              951.0        33827.0                           9.0   
1              120.0         3771.0                           2.0   
2              739.0        22327.0                           4.0   
3              289.0        10891.0                           1.0   

                                                                           \
category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                          646.0                1.0                 516.0   
1                           15.0                1.0                  15.0   
2                          220.0                5.0                 208.0   
3                          154.0                NaN                 105.0   

                                                                              \
category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0                   4.0       281.0            3870.0                14703.0   
1                   2.0        15.0             183.0                 4720.0   
2                   3.0       117.0            1632.0                13425.0   
3                   1.0        63.0             984.0                15729.0   

                                                                 \
category Spirituality & Religion Sports & Recreation Technology   
0                           20.0                 NaN     1148.0   
1                            2.0                 NaN      104.0   
2                           11.0                 1.0      669.0   
3                            5.0                 1.0      413.0   

                                                               
category Telecommunications  Trading Travel & Tourism Unknown  
0                      88.0  23164.0           1499.0  3566.0  
1                      10.0   1138.0             45.0   433.0  
2                      70.0   8707.0            537.0  1921.0  
3                      42.0   8488.0            473.0  2098.0

In [35]:
df.groupby(['taxi_income_segment','category']).agg(customers = ('userid', 'nunique')).reset_index()\
.pivot(index=['taxi_income_segment'], columns=['category'], values=['customers'])\
.reset_index().to_clipboard(index=False)

### NBS

In [36]:
def calculate_nbs(df, segment_column='gender'):
    
    # Step 1: Calculate total ads shown for each category
    category_totals = df.groupby(['category'])['aduuid'].count()

    # Step 2: Calculate total ads shown to each segment for each category
    segment_category_totals = df.groupby([segment_column, 'category'])['aduuid'].count()

    # Step 3: Calculate the overall total ads shown to each segment
    segment_totals = df.groupby(segment_column)['aduuid'].count()

    # Step 4: Calculate overall total ads
    total_ads = len(df['aduuid'])

    # Step 5: Calculate P(segment|category) and P(segment)
    P_segment_given_category = segment_category_totals / category_totals
    P_segment = segment_totals / total_ads

    # Step 6: Normalize the index for easy computation
    P_segment_given_category = P_segment_given_category.reset_index()
    P_segment = P_segment.reset_index()

    # Step 7: Merge P(segment|category) and P(segment) into a single DataFrame for NBS calculation
    nbs_df = P_segment_given_category.merge(P_segment, on=segment_column, suffixes=('_given_c', ''))

    # Step 8: Calculate NBS
    nbs_df['NBS'] = nbs_df['aduuid_given_c'] / nbs_df['aduuid']

    # Rename columns for clarity
    nbs_df = nbs_df.rename(columns={
        'category': 'Category',
        segment_column: 'Segment',
        'NBS': 'Normalized Bias Score'
    })

    return nbs_df[['Segment', 'Category', 'Normalized Bias Score']].pivot(index=['Segment'], columns=['Category'], values=['Normalized Bias Score']).reset_index()

In [37]:
calculate_nbs(df, segment_column='gender')

Segment Normalized Bias Score                                  \
Category                         Agriculture Automobile Automotive   Banking   
0               Female              0.901335   0.983768   1.605589  1.023257   
1                 Male              1.013888   1.018535   0.680759  1.000857   
2         Other/Unknow              0.620242   1.026745   1.156103  0.990914   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      2.506733          1.526104       1.215245     0.934238   
1                      0.190173          0.830756       0.940302     0.959670   
2                      1.879278               NaN       0.720278     1.010426   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         1.061638           1.095664      1.131155          2.394437   
1         1.083595           1.065072      0.911375          0.291120   
2              NaN                NaN      0.983449          1.406414   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         0.711431        1.063478         1.021764   1.897319      2.060240   
1         1.196575        0.943579         1.005816   0.628680      0.456916   
2         0.487023        1.077010         0.869454        NaN      1.799770   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           0.690160       0.762573                      0.782617   
1           1.178620       1.125149                      1.192881   
2           0.588985       0.996266                           NaN   

                                                                           \
Category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                       1.066353                NaN              2.156894   
1                       1.023554           1.661512              0.372993   
2                       0.973461                NaN              1.880576   

                                                                              \
Category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0              1.526104    1.030322          1.747354               1.078028   
1              0.593397    0.931853          0.611032               0.929877   
2              2.571101    2.133061          1.289563               0.953223   

                                                                 \
Category Spirituality & Religion Sports & Recreation Technology   
0                       1.459752                 NaN   0.717738   
1                       0.794636            1.661512   1.132453   
2                       0.521673                 NaN   0.836703   

                                                                  
Category Telecommunications   Trading Travel & Tourism   Unknown  
0                  1.218083  0.628113         0.970406  0.643326  
1                  0.933648  1.223923         1.039405  1.206163  
2                  0.330233  0.879687         0.916869  0.881399

In [38]:
calculate_nbs(df, segment_column='taxi_income_segment')

Segment Normalized Bias Score                                  \
Category                          Agriculture Automobile Automotive   Banking   
0           HIGH_INCOME              1.238837   1.054343   1.253771  1.133129   
1            LOW_INCOME              0.171278   0.939393   0.456295  0.781103   
2         MEDIUM_INCOME              0.302732   1.121626   0.845163  1.032520   
3               UNKNOWN              1.599472   0.747306   0.892373  0.762371   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      1.416126          0.738719       1.343871     0.365229   
1                      0.453989               NaN       0.426305     1.793513   
2                      0.798848          2.181067       0.646375     0.685716   
3                      0.705829          0.331190       1.087540     2.410925   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         1.507416           2.363902      0.801359          1.743342   
1              NaN                NaN      1.301971          0.351967   
2         0.809207                NaN      0.911917          0.400528   
3         0.691179                NaN      1.427282          0.666424   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         1.187766        1.027983         1.132975   0.383336      1.359244   
1         0.855125        0.745368         0.711857   2.749704      1.695651   
2         0.809114        1.023200         0.655107   1.697695      0.523456   
3         0.964646        0.923219         1.352792   1.002521      0.927332   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           1.000892       1.059919                      0.909193   
1           0.938064       0.906988                      5.652168   
2           1.182269       1.067625                      0.715837   
3           0.711823       0.778444                      0.271746   

                                                                           \
Category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                       1.459227           0.590976              1.478163   
1                       0.229286           1.413042              0.336823   
2                       0.728668           2.326471              0.842277   
3                       0.809853                NaN              0.513322   

                                                                              \
Category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0              1.181951    1.368312          1.413936               0.685031   
1              2.422358    0.448585          0.429531               1.580654   
2              0.747794    0.736716          0.764651               0.868519   
3              0.378503    0.748594          0.707353               1.629182   

                                                                 \
Category Spirituality & Religion Sports & Recreation Technology   
0                       1.267600                 NaN   1.048532   
1                       0.491493                 NaN   0.853090   
2                       0.960934            1.744853   0.971184   
3                       0.691179            2.649519   0.889219   

                                                                  
Category Telecommunications   Trading Travel & Tourism   Unknown  
0                  0.986767  1.267317         1.328116  1.031379  
1                  0.777821  0.457669         0.320019  0.851977  
2                  1.200587  0.723655         0.668741  0.820768  
3                  0.8629

In [44]:
calculate_nbs(df, segment_column='age_group')

Segment Normalized Bias Score                                  \
Category                     Agriculture Automobile Automotive   Banking   
0            21-25              0.690425   1.040097   0.894694  1.034328   
1            26-30              1.287524   0.975288   1.272995  0.987448   
2            31-35              1.517505   1.005983   1.321661  1.012033   
3            36-45              1.121841   1.031832   1.104008  1.065172   
4         Above-45              0.711743   1.051275   0.848267  1.043937   
5         Below-21              0.628208   1.129240   1.590807  1.098360   
6          UNKNOWN              0.736202   0.950507        NaN  0.962094   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      1.296280          0.820611       0.673667     0.804947   
1                      1.782483          2.279991       1.358870     0.725829   
2                      1.170335          0.613933       1.658154     0.714715   
3                      0.846895          0.819470       1.103278     0.977548   
4                      0.822882               NaN       0.695473     0.672503   
5                      0.613323               NaN       0.687129     0.823090   
6                      0.331735               NaN       0.254290     1.112983   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         0.380573           0.336661      0.930975          0.424834   
1         1.903297           2.619066      0.883461          1.331804   
2         0.284722           3.022438      0.914162          2.470724   
3         2.660307                NaN      1.064950          1.200688   
4              NaN                NaN      1.054101          1.501081   
5              NaN                NaN      0.838964               NaN   
6              NaN                NaN      0.897893          6.202708   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         0.629842        1.012081         0.829489   1.774294      0.164122   
1         1.211508        0.944758         1.124405   0.394377      0.364799   
2         1.286767        0.909124         1.230456   2.389360      0.982292   
3         0.899098        0.990211         1.039322   0.177183      2.294515   
4         1.802959        1.011302         1.008075        NaN           NaN   
5         0.173395        1.136498         0.815299        NaN           NaN   
6         1.066814        1.409678         1.736680        NaN           NaN   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           1.299269       0.984013                      0.504991   
1           0.816348       1.045657                      0.374152   
2           1.024082       1.042675                      0.503740   
3           0.707985       0.898140                      0.168096   
4           0.771243       0.880382                      0.808275   
5           1.241203       1.068272                      1.807307   
6           0.810959       0.990912                           NaN   

                                                                           \
Category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                       0.831965                NaN              0.282246   
1                       1.998362            2.43199              0.613457   
2                       1.538307                NaN              1.547612   
3                       0.777208                NaN              2.374344   
4                       0.842588                NaN              0.960692   
5                       0.509797                NaN     

In [65]:
calculate_nbs(df, segment_column='taxi_ltr_segment')

Segment Normalized Bias Score                                  \
Category                    Agriculture Automobile Automotive   Banking   
0              HH              0.458924   0.978225   0.683868  0.909242   
1             PHH              1.006881   1.010130   1.045229  1.019225   
2         UNKNOWN              0.427708   0.901999   0.089265  0.669503   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      0.610037               NaN       0.523103     1.241896   
1                      1.067537          1.132742       1.071716     0.916775   
2                      0.619477               NaN       0.422096     2.263115   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         1.343250                NaN      1.110343          0.358364   
1         1.017826           1.132742      0.976479          1.103697   
2         1.490336                NaN      1.287175          0.125559   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         0.902522        1.074466         0.770424   0.417497           NaN   
1         1.028245        0.982841         1.024377   1.102127      1.132742   
2         0.063243        0.928752         1.035539        NaN           NaN   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           1.391395       1.055785                      7.525645   
1           0.964693       0.987530                      0.551848   
2           1.041633       1.496459                           NaN   

                                                                           \
Category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                       0.568347           3.861844              0.466574   
1                       1.080735           0.849556              1.076644   
2                       0.161687                NaN              0.279818   

                                                                              \
Category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0              3.310152    0.711070          0.535246               1.179559   
1              0.809101    1.031454          1.061673               0.964633   
2                   NaN    0.108818          0.427948               1.170652   

                                                                 \
Category Spirituality & Religion Sports & Recreation Technology   
0                       0.447750                 NaN   1.278878   
1                       1.067075            1.132742   0.965923   
2                            NaN                 NaN   0.458419   

                                                                  
Category Telecommunications   Trading Travel & Tourism   Unknown  
0                  1.594339  0.737615         0.693165  0.837064  
1                  0.963870  1.033492         1.043641  1.016175  
2                  1.650991  0.692056         0.285007  1.020287

In [66]:
calculate_nbs(df, segment_column='taxi_retention_segments')

Segment Normalized Bias Score                                  \
Category                     Agriculture Automobile Automotive   Banking   
0          DORMANT              0.705089   1.036040   0.864767  0.438694   
1            ELITE              1.096885   1.025454   1.069148  0.979034   
2             GOLD              0.864180   0.997917   1.046557  1.170531   
3               HH              0.481937   0.951217   0.437672  1.093478   
4         INACTIVE              0.681986   1.017073   1.126296  0.467314   
5         PLATINUM              1.041903   0.997975   1.014770  1.082172   
6            PRIME              2.363833   1.041620   1.179214  0.983990   
7           SILVER              0.800548   0.978406   1.204637  1.246745   
8          UNKNOWN              0.427708   0.901999   0.089265  0.669503   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      0.882541          0.915636       0.526174     1.131745   
1                      1.551255          0.535814       1.332800     0.930525   
2                      0.542259               NaN       0.939341     0.909650   
3                      0.542905               NaN       0.584877     1.220877   
4                      0.233819               NaN       0.537699     1.243947   
5                      0.987120          1.967092       1.077337     0.853200   
6                      0.909273               NaN       0.697001     0.762663   
7                      0.477966          4.210380       0.622160     0.889541   
8                      0.619477               NaN       0.422096     2.263115   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         1.273928                NaN      1.200361          0.456141   
1         0.289909           1.978391      0.960508          1.920818   
2         0.647146           0.327129      0.951386          0.455644   
3              NaN                NaN      1.065616          0.267663   
4              NaN                NaN      1.156210          0.792120   
5         2.150361           1.152874      0.977538          0.721919   
6         1.406268                NaN      0.896021          0.236954   
7         2.733696                NaN      0.920322          0.427721   
8         1.490336                NaN      1.287175          0.125559   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         1.144263        0.990875         1.084534   0.395951      0.366254   
1         0.994738        0.980497         1.116276   1.235751      1.643163   
2         1.145554        0.972704         0.899320   1.724057      0.159475   
3         0.970700        1.122744         0.727887   0.592477           NaN   
4         1.196949        1.066406         0.842082        NaN      1.247589   
5         0.918045        0.969198         1.003747   0.850634      1.124053   
6         1.014485        0.899660         0.884694        NaN      7.277435   
7         1.044050        1.031711         0.891064   0.728282      0.336830   
8         0.063243        0.928752         1.035539        NaN           NaN   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           1.440589       1.110537                      0.751291   
1           0.811835       0.907201                      0.512916   
2           1.117346       1.060835                      1.308515   
3           1.356224       1.012204                     10.679773   
4           1.555355       1.158573                           NaN   
5           0.847813       0.985789                      0.230575   
6           0.846785       0.945675          

In [67]:
calculate_nbs(df, segment_column='taxi_regularity_segment')

Segment Normalized Bias Score                                  \
Category                      Agriculture Automobile Automotive   Banking   
0         BI_WEEKLY              0.909936   0.994940   0.992272  1.075774   
1             DAILY              1.141417   1.050496   1.240119  0.936688   
2           MONTHLY              0.847419   0.990859   1.196059  1.011259   
3         RARE_NEED              0.779790   0.994596   0.630955  0.754811   
4           UNKNOWN              0.454847   0.968270   0.606213  0.877932   
5            WEEKLY              1.018937   0.986859   0.907457  1.116671   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      0.578420          1.367341       0.796972     0.978909   
1                      1.689660          0.650159       1.391277     0.923529   
2                      0.380231          2.266218       0.731763     1.004267   
3                      0.851409          1.261910       0.486894     0.970308   
4                      0.611270               NaN       0.509911     1.375266   
5                      0.987285          1.118783       1.126016     0.833723   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         2.377984           0.560960      0.980271          0.497518   
1         0.351777           3.111874      0.930964          1.812085   
2         1.050999                NaN      0.998358          0.354183   
3              NaN                NaN      1.064639          0.443749   
4         1.362459                NaN      1.133437          0.327960   
5         0.985826                NaN      0.998876          1.108128   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         1.039381        0.969763         0.912119   1.034744      0.410202   
1         0.983103        0.984159         1.099451   1.405750      1.733758   
2         1.107557        1.033568         1.036197   0.326662      0.906487   
3         0.831960        1.128431         0.982258   0.545691      0.504764   
4         0.792913        1.055436         0.805048   0.362972           NaN   
5         1.078876        0.949198         1.024314   1.161115      1.163534   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           1.119101       1.073374                      0.420720   
1           0.803522       0.906437                      0.978018   
2           1.224282       1.053422                           NaN   
3           1.176760       1.100238                      1.035413   
4           1.345716       1.113337                      6.542800   
5           0.915530       0.975561                      0.275393   

                                                                           \
Category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                       0.822124           1.367341              0.881051   
1                       1.480782                NaN              0.934107   
2                       0.691743                NaN              1.016252   
3                       1.200005           3.365093              1.032878   
4                       0.515237           3.357489              0.442184   
5                       0.931053           1.193369              1.377488   

                                                                              \
Category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0              0.390669    0.743718          0.741965               0.994918   
1              1.981438    1.179690          1.278967               0.918174   
2                   NaN    0.9

In [68]:
calculate_nbs(df, segment_column='ps_tag_link')

Segment Normalized Bias Score                                  \
Category                    Agriculture Automobile Automotive   Banking   
0             NPS              0.866365   1.009300   0.944625  1.014328   
1              PS              1.299260   1.019770   0.949069  0.972360   
2         UNKNOWN              0.926546   1.003557   1.050986  1.020603   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      0.764267          0.335039       0.715118     1.012417   
1                      1.018603          1.385127       1.197743     0.885884   
2                      1.121193          1.188549       1.087928     0.945217   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         0.466142                NaN      0.928681          0.431992   
1         1.927133           3.788382      0.904450          1.046315   
2         1.033521           0.772049      1.020747          1.230635   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         1.216527        0.926029         1.028941        NaN      1.474174   
1         0.944999        0.959729         1.014334   0.598974      0.554051   
2         0.969747        1.013687         0.998693   1.456241      1.030076   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           1.041718       1.056617                      0.549808   
1           1.041849       1.045025                      0.189419   
2           0.969806       0.968682                      1.340928   

                                                                           \
Category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                       0.950754                NaN              0.446355   
1                       1.470500           3.078060              0.365849   
2                       0.975757           0.924427              1.344974   

                                                                              \
Category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0              0.765805    0.689224          0.754131               0.958232   
1              2.638337    1.614272          1.047601               0.928749   
2              0.679171    0.966771          1.095556               0.996797   

                                                                 \
Category Spirituality & Religion Sports & Recreation Technology   
0                       0.155381            2.680316   0.966979   
1                       0.535315                 NaN   1.067936   
2                       1.378028            0.792366   0.971377   

                                                                  
Category Telecommunications   Trading Travel & Tourism   Unknown  
0                  0.959012  1.262740         0.943788  1.179659  
1                  0.931890  1.232992         1.166064  1.132322  
2                  1.043161  0.889998         1.006013  0.923747

In [69]:
calculate_nbs(df, segment_column='ps_tag_auto')

Segment Normalized Bias Score                                  \
Category                    Agriculture Automobile Automotive   Banking   
0             NPS              1.053784   1.008516   1.083801  1.014029   
1              PS              0.930456   1.019975   1.056481  1.005862   
2         UNKNOWN              0.861394   0.992551   0.866999  1.016178   

                                                                               \
Category Beauty & Personal Care Business Services Consumer Goods DIY & Crafts   
0                      1.361262          0.855945       1.257787     0.877025   
1                      0.909677          1.014221       1.005375     1.013083   
2                      0.623628          1.397551       0.682642     1.010999   

                                                                       \
Category Education Energy & Utilities Entertainment Fashion & Apparel   
0         1.223960           1.346102      0.992088          1.517500   
1         0.470364           1.664363      0.962157          0.866856   
2         1.296279                NaN      0.998688          0.448217   

                                                                              \
Category   Finance Food & Beverage Gaming & Esports Healthcare Home & Garden   
0         0.866123        1.009456         1.084962   1.418864      1.255386   
1         1.217565        0.966435         0.997314   0.877165      0.912799   
2         1.070367        0.975840         0.888022   0.604346      0.838531   

                                                                   \
Category Investments Legal Services Logistics & Delivery Services   
0           0.903120       0.907425                      0.468209   
1           1.013431       1.023096                      1.040227   
2           1.124952       1.118362                      1.911181   

                                                                           \
Category Luxury Goods & Services Media & Publishing Parenting & Childcare   
0                       1.309219                NaN              1.209270   
1                       0.864959           1.352295              0.903002   
2                       0.765284           2.484535              0.853869   

                                                                              \
Category Pets & Animals Real Estate Retail/E-commerce Social Network & Media   
0              0.652149    1.129183          1.254060               0.967294   
1              1.159110    1.015295          0.890581               0.965237   
2              1.331001    0.790713          0.763500               1.012820   

                                                                 \
Category Spirituality & Religion Sports & Recreation Technology   
0                       1.554760                 NaN   0.934667   
1                       0.823136            4.056886   0.942365   
2                       0.324070                 NaN   1.103623   

                                                                  
Category Telecommunications   Trading Travel & Tourism   Unknown  
0                  1.005147  1.014375         1.094417  1.013147  
1                  1.116574  1.052274         1.003915  1.031573  
2                  0.923153  0.970019         0.901083  0.961863

In [45]:
df_category.head(1)

,hash_id,advertiserName,title,body,category
0,8bf4cf10f139c08f50c0042e965d7f63,,Prime Video,New On Prime - Bloody Beggar A beggar's life t...,Entertainment


In [66]:
df_category[df_category['category'] == 'Education'].head(50)

,hash_id,advertiserName,title,body,category
540,86d38a5428263edc2901bf924a8c0cca,edureka.co,Make a Move with Edureka,Get exclusive offers & discounts on the world'...,Education
651,0d496cbecf3251d10b3844b495327afb,Element14,Electronic Components,Learn more about Schneider’s Top TeSys island ...,Education
786,1ee7afa6df8ff24173bed8c62d9d1ba8,Blinkit,Doms Whiteboard Duster & Marker...,₹150 Doms white board magnetic duster and 2 ma...,Education
896,eaf5f0b97e0b5f06565ece6b5bfcceef,Element14,80+ Years Industry Experience,Learn more about Schneider’s Top TeSys island ...,Education
1198,2d37cf545a746790b6e17acd993110cb,KPMG Learning Academy,Live Interactive IFRS Sessions,75 Hrs intensive IFRS sessions on weekends. AC...,Education
1639,3f04853aea18a17cfa0ec7ca7366f290,edureka.co,Make a Move with Edureka,World-Class Instructors,Education
1652,09b16e04d1c0b31ff8b0af868c5f9668,,Kids Study Tables - Ko...,Rs. 8,Education
2366,112e4ef0da65dffd3489339a0b897f90,,Unacademy: Learn & Crack Exams,CAT Batches by Top Educators One-stop solution...,Education
3015,db69137a3f42f59e68d3647b13b402e9,,Unacademy: Learn & Crack Exams,Top ranking IIT/NEET Coaching Crack IIT/NEET w...,Education
3368,40355c4b9b79d9ad5980770a269e0698,,Unacademy: Learn & Crack Exams,CAT 2025 Batch Comprehensive Batch for CAT 202...,Education


In [56]:
df_category[['category', 'advertiserName']].drop_duplicates().to_clipboard(index=False)